# Set 1 (`c1_M0`) EF5 PET/CT & RT_Image ROI Interactive Explorer

This notebook loads the registered DICOM volumes (`c1_M0_PET_mouse.dcm` and `c1_M0_CT_mouse.dcm`) and the binary RT_Image ROI archive (`rois.ROI`), computes quantitative ROI summary statistics, and provides interactive slice viewers across all three orthogonal planes (**Transaxial**, **Coronal**, and **Sagittal**).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.rt_image_loader import (
    load_rt_image_dicom,
    load_rt_image_roi,
    build_roi_3d_mask,
    compute_roi_statistics,
)
from src.inspect_set1 import ROI_STYLES, get_slice_2d_and_extent, plot_rois_on_ax

root = Path('/workspace') if Path('/workspace').exists() else Path('..').resolve()
set1_dir = root / 'images' / 'set1'

pet_vol = load_rt_image_dicom(set1_dir / 'c1_M0_PET_mouse.dcm')
ct_vol = load_rt_image_dicom(set1_dir / 'c1_M0_CT_mouse.dcm')
roi_version, rois = load_rt_image_roi(set1_dir / 'rois.ROI')
roi_masks_ct = {r.name: build_roi_3d_mask(ct_vol, r) for r in rois}

print(f'PET shape: {pet_vol.shape}, spacing: {pet_vol.spacing} mm, origin: {pet_vol.origin}')
print(f'CT  shape: {ct_vol.shape}, spacing: {ct_vol.spacing} mm, origin: {ct_vol.origin}')
print(f'ROIs ({roi_version}): {[r.name for r in rois]}')

## Quantitative ROI Summary Statistics

In [ ]:
stats_df = compute_roi_statistics([pet_vol, ct_vol], rois)
bg_mean = float(stats_df[(stats_df['Modality'] == 'PT') & (stats_df['ROI Label'] == 'M0 bg ROI')]['Mean'].iloc[0])
stats_df['TBR_mean'] = np.where(stats_df['Modality'] == 'PT', (stats_df['Mean'] / bg_mean).round(3), np.nan)
stats_df['TBR_max'] = np.where(stats_df['Modality'] == 'PT', (stats_df['Maximum'] / bg_mean).round(3), np.nan)
stats_df

## Interactive Multi-Planar PET/CT + ROI Slice Viewer

In [ ]:
def interactive_slice_viewer(plane='transaxial', coord_mm=-58.11, pet_max=2.2, alpha=0.6, show_rois=True):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), facecolor='#111111')
    ct_2d, ct_ext, ct_idx, _ = get_slice_2d_and_extent(ct_vol, plane, coord_mm)
    pet_2d, pet_ext, pet_idx, _ = get_slice_2d_and_extent(pet_vol, plane, coord_mm)
    
    for idx, (ax, title) in enumerate(zip(axes, [f'CT (slice {ct_idx})', f'EF5 PET (slice {pet_idx})', 'PET/CT Overlay'])):
        ax.set_facecolor('black')
        if idx == 0:
            ax.imshow(ct_2d, cmap='gray', vmin=-500, vmax=800, extent=ct_ext, origin='upper', aspect='equal')
        elif idx == 1:
            ax.imshow(pet_2d, cmap='inferno', vmin=0, vmax=pet_max, extent=pet_ext, origin='upper', aspect='equal')
        else:
            ax.imshow(ct_2d, cmap='gray', vmin=-500, vmax=800, extent=ct_ext, origin='upper', aspect='equal')
            ax.imshow(np.ma.masked_less(pet_2d, 0.05), cmap='inferno', vmin=0, vmax=pet_max, extent=pet_ext, origin='upper', alpha=alpha, aspect='equal')
        if show_rois:
            plot_rois_on_ax(ax, plane, coord_mm, rois, ct_vol, roi_masks_ct)
        ax.set_xlim(ct_ext[0], ct_ext[1])
        ax.set_ylim(ct_ext[2], ct_ext[3])
        ax.set_title(title, color='white')
        ax.tick_params(colors='white')
    plt.suptitle(f'{plane.capitalize()} @ {coord_mm:.2f} mm', color='white', fontsize=13)
    plt.tight_layout()
    plt.show()

widgets.interact(
    interactive_slice_viewer,
    plane=widgets.Dropdown(options=['transaxial', 'coronal', 'sagittal'], value='transaxial'),
    coord_mm=widgets.FloatSlider(min=-82.0, max=-48.0, step=0.4, value=-58.11, description='Coord (mm)'),
    pet_max=widgets.FloatSlider(min=0.5, max=5.0, step=0.1, value=2.2, description='PET Max'),
    alpha=widgets.FloatSlider(min=0.1, max=1.0, step=0.05, value=0.6, description='Overlay Alpha'),
    show_rois=widgets.Checkbox(value=True, description='Show ROIs')
);